In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy import stats
from scipy.stats import chi2_contingency, kruskal

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 200)
pd.set_option("display.width", 160)
pd.set_option("display.float_format", lambda x: f"{x:.3f}")

In [ ]:
FILE_PATH = Path(
    "../data/raw/張瓊之_給馬老師資料20250925_V4(遺失值不用登錄成-1).xlsx"
)

OUTPUT_DIR = Path("../results/descriptive_statistics")
FIGURE_DIR = OUTPUT_DIR / "figures"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

if not FILE_PATH.exists():
    raise FileNotFoundError(f"找不到資料檔案：{FILE_PATH.resolve()}")

print("資料檔案：", FILE_PATH.resolve())
print("輸出資料夾：", OUTPUT_DIR.resolve())

In [ ]:
demo = pd.read_excel(FILE_PATH, sheet_name="demographic")
mmse = pd.read_excel(FILE_PATH, sheet_name="MMSE longitudinal")
casi = pd.read_excel(FILE_PATH, sheet_name="CASI longitudinal")

print("demographic shape:", demo.shape)
print("MMSE shape:", mmse.shape)
print("CASI shape:", casi.shape)

In [ ]:
demo = demo.iloc[1:].copy()

demo.columns = [
    "patient_id",
    "diagnosis",
    "bio_category",
    "apoe",
    "gender",
    "age_first_mmse",
    "education",
    "hypertension",
    "diabetes",
    "hyperlipidemia"
]

demo.head()

In [ ]:
mmse = mmse.rename(columns={
    "Count Number": "patient_id",
    "Date": "visit_date",
    "MMSE(numerical)": "mmse",
    "CDR (scale)": "cdr_global",
    "CDR_M (scale)": "cdr_memory",
    "CDR_O (scale)": "cdr_orientation",
    "CDR_J (scale)": "cdr_judgment",
    "CDR_C (scale)": "cdr_community",
    "CDR_H (scale)": "cdr_home_hobbies",
    "CDR_P (scale)": "cdr_personal_care",
    "CDR-SOB(numerical)": "cdr_sob"
})

mmse.head()

In [ ]:
casi = casi.rename(columns={
    "Count Number": "patient_id",
    "Date": "visit_date",
    "MENMA10": "casi_mental_manipulation",
    "ATTEN8": "casi_attention",
    "ORIEN18": "casi_orientation",
    "LTM10": "casi_long_term_memory",
    "STM12": "casi_short_term_memory",
    "ABSTR12": "casi_abstraction",
    "DRAW10": "casi_drawing",
    "ANML10": "casi_verbal_fluency",
    "LANG10": "casi_language",
    "Total": "casi_total"
})

casi.head()

## 統一 ID、日期及數值格式

In [ ]:
# 統一 patient_id
for df in [demo, mmse, casi]:
    df["patient_id"] = pd.to_numeric(
        df["patient_id"],
        errors="coerce"
    ).astype("Int64")

    df.dropna(subset=["patient_id"], inplace=True)

# 日期格式
mmse["visit_date"] = pd.to_datetime(
    mmse["visit_date"],
    errors="coerce"
)

casi["visit_date"] = pd.to_datetime(
    casi["visit_date"],
    errors="coerce"
)

# 基本資料數值欄位
demo_numeric_cols = [
    "gender",
    "age_first_mmse",
    "education",
    "hypertension",
    "diabetes",
    "hyperlipidemia"
]

for col in demo_numeric_cols:
    demo[col] = pd.to_numeric(demo[col], errors="coerce")

# MMSE/CDR 數值欄位
mmse_numeric_cols = [
    "mmse",
    "cdr_global",
    "cdr_memory",
    "cdr_orientation",
    "cdr_judgment",
    "cdr_community",
    "cdr_home_hobbies",
    "cdr_personal_care",
    "cdr_sob"
]

for col in mmse_numeric_cols:
    mmse[col] = pd.to_numeric(mmse[col], errors="coerce")

# CASI 數值欄位
casi_numeric_cols = [
    "casi_mental_manipulation",
    "casi_attention",
    "casi_orientation",
    "casi_long_term_memory",
    "casi_short_term_memory",
    "casi_abstraction",
    "casi_drawing",
    "casi_verbal_fluency",
    "casi_language",
    "casi_total"
]

for col in casi_numeric_cols:
    casi[col] = pd.to_numeric(casi[col], errors="coerce")

# -1 視為缺失
for df in [demo, mmse, casi]:
    df.replace(-1, np.nan, inplace=True)

print("完成型別轉換")

In [ ]:
print("Biological category 原始值：")
display(
    demo["bio_category"]
    .astype("string")
    .value_counts(dropna=False)
    .to_frame("n")
)

In [ ]:
def clean_bio_category(value):
    if pd.isna(value):
        return np.nan

    value = str(value).strip()
    value_upper = value.upper().replace("_", "-").replace(" ", "")

    mapping = {
        "AD": "AD",
        "NON-AD": "non-AD",
        "NONAD": "non-AD",
        "CU": "CU"
    }

    return mapping.get(value_upper, value)


demo["bio_category"] = demo["bio_category"].apply(clean_bio_category)

GROUP_ORDER = ["AD", "non-AD", "CU"]

print("清理後 Biological category：")
display(
    demo["bio_category"]
    .value_counts(dropna=False)
    .reindex(GROUP_ORDER)
    .to_frame("病患數")
)

In [ ]:
# 清理類別欄位與 APOE
def clean_apoe(value):
    if pd.isna(value):
        return np.nan

    value = str(value).upper().strip()
    value = value.replace(" ", "")
    value = value.replace("Ε", "E")

    return value


demo["apoe_clean"] = demo["apoe"].apply(clean_apoe)

demo["apoe4_count"] = demo["apoe_clean"].apply(
    lambda x: np.nan if pd.isna(x) else x.count("E4")
)

demo["apoe4_carrier"] = np.where(
    demo["apoe4_count"].isna(),
    np.nan,
    (demo["apoe4_count"] >= 1).astype(int)
)

# 只保留教授指定三組
demo = demo[
    demo["bio_category"].isin(GROUP_ORDER)
].copy()

demo["bio_category"] = pd.Categorical(
    demo["bio_category"],
    categories=GROUP_ORDER,
    ordered=True
)

print("三組病患數：")
display(
    demo.groupby("bio_category", observed=True)["patient_id"]
    .nunique()
    .to_frame("病患數")
)

In [ ]:
# 合理範圍檢查
# 只將明顯超出量表範圍的值設成缺失，不做補值。

def set_outside_range_to_nan(df, column, lower, upper):
    invalid = (
        df[column].notna()
        & ~df[column].between(lower, upper)
    )

    print(
        f"{column}: {invalid.sum()} 筆超出 [{lower}, {upper}]，改為缺失"
    )

    df.loc[invalid, column] = np.nan


# MMSE
set_outside_range_to_nan(mmse, "mmse", 0, 30)

# CDR global
set_outside_range_to_nan(mmse, "cdr_global", 0, 3)

# CDR-SOB
set_outside_range_to_nan(mmse, "cdr_sob", 0, 18)

# CASI total
set_outside_range_to_nan(casi, "casi_total", 0, 100)

# 基本資料
set_outside_range_to_nan(demo, "age_first_mmse", 18, 110)
set_outside_range_to_nan(demo, "education", 0, 30)

In [ ]:
# 處理同病患、同日期重複紀錄
# 不要先刪除量表缺失，只排除沒有 ID 或日期的紀錄。同日多筆以平均值代表。
mmse = mmse.dropna(
    subset=["patient_id", "visit_date"]
).copy()

casi = casi.dropna(
    subset=["patient_id", "visit_date"]
).copy()


mmse = (
    mmse
    .groupby(
        ["patient_id", "visit_date"],
        as_index=False
    )[mmse_numeric_cols]
    .mean()
)


casi = (
    casi
    .groupby(
        ["patient_id", "visit_date"],
        as_index=False
    )[casi_numeric_cols]
    .mean()
)

print("去除同日重複後：")
print("MMSE/CDR:", mmse.shape)
print("CASI:", casi.shape)

In [ ]:
# 合併 Biological category 與基本資料
# MMSE/CDR 和 CASI 分開建立 longitudinal stack data，不要為了畫圖先把兩張量表硬合成同一日期。

demo_features = [
    "patient_id",
    "diagnosis",
    "bio_category",
    "gender",
    "age_first_mmse",
    "education",
    "hypertension",
    "diabetes",
    "hyperlipidemia",
    "apoe_clean",
    "apoe4_count",
    "apoe4_carrier"
]

mmse_long = mmse.merge(
    demo[demo_features],
    on="patient_id",
    how="inner",
    validate="many_to_one"
)

casi_long = casi.merge(
    demo[demo_features],
    on="patient_id",
    how="inner",
    validate="many_to_one"
)

print("MMSE/CDR longitudinal shape:", mmse_long.shape)
print("CASI longitudinal shape:", casi_long.shape)

In [ ]:
# 建立追蹤時間 
# MMSE/CDR 以第一次 MMSE/CDR 日期為基準；CASI 以第一次 CASI 日期為基準。

def add_followup_time(df):
    df = df.sort_values(
        ["patient_id", "visit_date"]
    ).copy()

    df["baseline_date"] = (
        df.groupby("patient_id")["visit_date"]
        .transform("min")
    )

    df["time_since_baseline_days"] = (
        df["visit_date"] - df["baseline_date"]
    ).dt.days

    df["time_since_baseline_months"] = (
        df["time_since_baseline_days"] / 30.4375
    )

    df["time_since_baseline_years"] = (
        df["time_since_baseline_days"] / 365.25
    )

    df["visit_number"] = (
        df.groupby("patient_id").cumcount() + 1
    )

    return df


mmse_long = add_followup_time(mmse_long)
casi_long = add_followup_time(casi_long)

display(
    mmse_long[
        [
            "patient_id",
            "bio_category",
            "visit_date",
            "visit_number",
            "time_since_baseline_years",
            "mmse",
            "cdr_global"
        ]
    ].head(10)
)

In [ ]:
# 先檢查追蹤結構
# 這一步會幫你決定要用 3 個月、6 個月還是 12 個月切區間。

def followup_structure(df, name):
    patient_summary = (
        df.groupby(
            ["patient_id", "bio_category"],
            observed=True
        )
        .agg(
            n_visits=("visit_date", "nunique"),
            followup_years=("time_since_baseline_years", "max")
        )
        .reset_index()
    )

    print(f"\n{name}")
    print("病患數：", patient_summary["patient_id"].nunique())
    print("總觀測數：", len(df))

    display(
        patient_summary.groupby(
            "bio_category",
            observed=True
        ).agg(
            patient_n=("patient_id", "nunique"),
            visits_median=("n_visits", "median"),
            visits_q1=("n_visits", lambda x: x.quantile(0.25)),
            visits_q3=("n_visits", lambda x: x.quantile(0.75)),
            followup_median=("followup_years", "median"),
            followup_q1=("followup_years", lambda x: x.quantile(0.25)),
            followup_q3=("followup_years", lambda x: x.quantile(0.75))
        )
    )

    return patient_summary


mmse_followup = followup_structure(mmse_long, "MMSE/CDR")
casi_followup = followup_structure(casi_long, "CASI")

In [ ]:
# 建立時間區間
# 先使用每 6 個月一期。之後若後期人數太少，可以改成 12 個月

""" 
T1：0 至未滿 6 個月
T2：6 至未滿 12 個月
T3：12 至未滿 18 個月"""

BIN_MONTHS = 12

def add_time_category(df, bin_months=6):
    df = df.copy()

    bin_number = np.floor(
        df["time_since_baseline_months"] / bin_months
    ).astype("Int64") + 1

    df["time_category_number"] = bin_number
    df["time_category"] = "T" + bin_number.astype(str)

    df["time_mid_years"] = (
        (bin_number - 0.5) * bin_months / 12
    )

    return df


mmse_long = add_time_category(mmse_long, BIN_MONTHS)
casi_long = add_time_category(casi_long, BIN_MONTHS)

print(
    mmse_long["time_category"]
    .value_counts()
    .sort_index()
)

In [ ]:
# 產生三組樣本與觀測數檢查表

group_count_table = (
    demo.groupby("bio_category", observed=True)
    .agg(
        patient_n=("patient_id", "nunique")
    )
    .reset_index()
)

mmse_observations = (
    mmse_long.groupby("bio_category", observed=True)
    .agg(
        mmse_cdr_observation_n=("visit_date", "size"),
        mmse_patient_n=("patient_id", "nunique")
    )
    .reset_index()
)

casi_observations = (
    casi_long.groupby("bio_category", observed=True)
    .agg(
        casi_observation_n=("visit_date", "size"),
        casi_patient_n=("patient_id", "nunique")
    )
    .reset_index()
)

group_count_table = (
    group_count_table
    .merge(mmse_observations, on="bio_category", how="left")
    .merge(casi_observations, on="bio_category", how="left")
)

display(group_count_table)

In [ ]:
# 建立每位病患的基準資料
# 表 4.1 必須一位病患一列，不能讓追蹤次數多的人重複出現。

# 第一次 MMSE/CDR 觀測
baseline_mmse = (
    mmse_long
    .sort_values(["patient_id", "visit_date"])
    .groupby("patient_id", as_index=False)
    .first()[
        [
            "patient_id",
            "mmse",
            "cdr_global",
            "cdr_sob"
        ]
    ]
    .rename(columns={
        "mmse": "baseline_mmse",
        "cdr_global": "baseline_cdr_global",
        "cdr_sob": "baseline_cdr_sob"
    })
)

# 第一次 CASI 觀測
baseline_casi = (
    casi_long
    .sort_values(["patient_id", "visit_date"])
    .groupby("patient_id", as_index=False)
    .first()[
        ["patient_id", "casi_total"]
    ]
    .rename(columns={
        "casi_total": "baseline_casi"
    })
)

# 每位病患追蹤結構
patient_mmse_structure = (
    mmse_long.groupby("patient_id")
    .agg(
        mmse_cdr_visit_n=("visit_date", "nunique"),
        mmse_cdr_followup_years=("time_since_baseline_years", "max")
    )
    .reset_index()
)

patient_casi_structure = (
    casi_long.groupby("patient_id")
    .agg(
        casi_visit_n=("visit_date", "nunique"),
        casi_followup_years=("time_since_baseline_years", "max")
    )
    .reset_index()
)

baseline_df = (
    demo
    .merge(baseline_mmse, on="patient_id", how="left")
    .merge(baseline_casi, on="patient_id", how="left")
    .merge(patient_mmse_structure, on="patient_id", how="left")
    .merge(patient_casi_structure, on="patient_id", how="left")
)

print("基準資料病患數：", baseline_df["patient_id"].nunique())
display(baseline_df.head())

In [ ]:
# 表 4.1 的格式化函數
def format_mean_sd(series, digits=1):
    values = pd.to_numeric(series, errors="coerce").dropna()

    if len(values) == 0:
        return "NA"

    return f"{values.mean():.{digits}f} ± {values.std(ddof=1):.{digits}f}"


def format_median_iqr(series, digits=1):
    values = pd.to_numeric(series, errors="coerce").dropna()

    if len(values) == 0:
        return "NA"

    q1 = values.quantile(0.25)
    median = values.median()
    q3 = values.quantile(0.75)

    return (
        f"{median:.{digits}f} "
        f"({q1:.{digits}f}, {q3:.{digits}f})"
    )


def format_n_percent(series, target=1, digits=1):
    valid = series.dropna()

    if len(valid) == 0:
        return "NA"

    n = (valid == target).sum()
    percent = 100 * n / len(valid)

    return f"{n} ({percent:.{digits}f}%)"


def kruskal_pvalue(df, variable):
    groups = [
        df.loc[
            df["bio_category"] == group,
            variable
        ].dropna()
        for group in GROUP_ORDER
    ]

    groups = [x for x in groups if len(x) > 0]

    if len(groups) < 2:
        return np.nan

    try:
        return kruskal(*groups).pvalue
    except ValueError:
        return np.nan


def chi_square_pvalue(df, variable):
    table = pd.crosstab(
        df["bio_category"],
        df[variable]
    )

    if table.shape[0] < 2 or table.shape[1] < 2:
        return np.nan

    try:
        return chi2_contingency(table)[1]
    except ValueError:
        return np.nan


def format_pvalue(p):
    if pd.isna(p):
        return "NA"

    if p < 0.001:
        return "<0.001"

    return f"{p:.3f}"

In [ ]:
# 建立表 4.1

table1_rows = []


def add_continuous_row(
    label,
    variable,
    summary_type="mean_sd",
    digits=1
):
    if summary_type == "mean_sd":
        formatter = lambda x: format_mean_sd(x, digits)
    else:
        formatter = lambda x: format_median_iqr(x, digits)

    row = {
        "變數": label,
        "整體": formatter(baseline_df[variable])
    }

    for group in GROUP_ORDER:
        group_data = baseline_df.loc[
            baseline_df["bio_category"] == group,
            variable
        ]

        row[group] = formatter(group_data)

    row["p 值"] = format_pvalue(
        kruskal_pvalue(baseline_df, variable)
    )

    table1_rows.append(row)


def add_binary_row(label, variable, target=1):
    row = {
        "變數": label,
        "整體": format_n_percent(
            baseline_df[variable],
            target=target
        )
    }

    for group in GROUP_ORDER:
        group_data = baseline_df.loc[
            baseline_df["bio_category"] == group,
            variable
        ]

        row[group] = format_n_percent(
            group_data,
            target=target
        )

    row["p 值"] = format_pvalue(
        chi_square_pvalue(baseline_df, variable)
    )

    table1_rows.append(row)


# 病患人數
n_row = {
    "變數": "病患人數",
    "整體": str(baseline_df["patient_id"].nunique()),
    "p 值": ""
}

for group in GROUP_ORDER:
    n_row[group] = str(
        baseline_df.loc[
            baseline_df["bio_category"] == group,
            "patient_id"
        ].nunique()
    )

table1_rows.append(n_row)

# 基本特徵
add_continuous_row("第一次 MMSE 年齡（歲）", "age_first_mmse")
add_binary_row("男性，n (%)", "gender", target=1)
add_continuous_row("教育年數", "education")
add_binary_row("APOE ε4 carrier，n (%)", "apoe4_carrier", target=1)
add_binary_row("高血壓，n (%)", "hypertension", target=1)
add_binary_row("糖尿病，n (%)", "diabetes", target=1)
add_binary_row("高血脂，n (%)", "hyperlipidemia", target=1)

# 基準量表
add_continuous_row("基準 MMSE", "baseline_mmse")
add_continuous_row("基準 CASI", "baseline_casi")
add_continuous_row(
    "基準 CDR global",
    "baseline_cdr_global",
    summary_type="median_iqr"
)
add_continuous_row(
    "基準 CDR-SOB",
    "baseline_cdr_sob",
    summary_type="median_iqr"
)

# 追蹤結構
add_continuous_row(
    "MMSE/CDR 測量次數",
    "mmse_cdr_visit_n",
    summary_type="median_iqr"
)
add_continuous_row(
    "MMSE/CDR 追蹤時間（年）",
    "mmse_cdr_followup_years",
    summary_type="median_iqr"
)
add_continuous_row(
    "CASI 測量次數",
    "casi_visit_n",
    summary_type="median_iqr"
)
add_continuous_row(
    "CASI 追蹤時間（年）",
    "casi_followup_years",
    summary_type="median_iqr"
)

table1 = pd.DataFrame(table1_rows)

display(table1)

In [ ]:
# 建立各時間點量表統計函數

def create_timepoint_summary(
    df,
    outcome,
    digits=1
):
    rows = []

    category_numbers = sorted(
        df["time_category_number"]
        .dropna()
        .astype(int)
        .unique()
    )

    for category_number in category_numbers:
        temp = df[
            df["time_category_number"] == category_number
        ].copy()

        available_n = temp[outcome].notna().sum()
        missing_n = temp[outcome].isna().sum()
        total_n = len(temp)

        missing_percent = (
            100 * missing_n / total_n
            if total_n > 0
            else np.nan
        )

        row = {
            "時間點": f"T{category_number}",
            "時間範圍（月）": (
                f"{(category_number - 1) * BIN_MONTHS}–"
                f"<{category_number * BIN_MONTHS}"
            ),
            "可用 N": int(available_n),
            "缺失 N (%)": (
                f"{missing_n} ({missing_percent:.1f}%)"
            ),
            "整體": format_mean_sd(
                temp[outcome],
                digits=digits
            )
        }

        for group in GROUP_ORDER:
            group_values = temp.loc[
                temp["bio_category"] == group,
                outcome
            ]

            row[group] = format_mean_sd(
                group_values,
                digits=digits
            )

        row["p 值"] = format_pvalue(
            kruskal_pvalue(temp, outcome)
        )

        rows.append(row)

    return pd.DataFrame(rows)

In [ ]:
# 建立 MMSE、CASI、CDR 表

table_mmse = create_timepoint_summary(
    mmse_long,
    outcome="mmse",
    digits=1
)

table_casi = create_timepoint_summary(
    casi_long,
    outcome="casi_total",
    digits=1
)

table_cdr = create_timepoint_summary(
    mmse_long,
    outcome="cdr_global",
    digits=2
)

table_cdr_sob = create_timepoint_summary(
    mmse_long,
    outcome="cdr_sob",
    digits=1
)

print("表 4.2 型：MMSE")
display(table_mmse)

print("表 4.3 型：CASI")
display(table_casi)

print("表 4.4 型：CDR global")
display(table_cdr)

print("補充表：CDR-SOB")
display(table_cdr_sob)

In [ ]:
# 檢查各時間點三組人數
# 後期樣本太少時，平均線會非常不穩定。

def count_by_time_and_group(df, outcome):
    return (
        df.dropna(subset=[outcome])
        .groupby(
            ["time_category", "bio_category"],
            observed=True
        )["patient_id"]
        .nunique()
        .unstack(fill_value=0)
        .reindex(columns=GROUP_ORDER)
    )


mmse_n_table = count_by_time_and_group(
    mmse_long,
    "mmse"
)

casi_n_table = count_by_time_and_group(
    casi_long,
    "casi_total"
)

cdr_n_table = count_by_time_and_group(
    mmse_long,
    "cdr_global"
)

print("MMSE 各時間點病患數")
display(mmse_n_table)

print("CASI 各時間點病患數")
display(casi_n_table)

print("CDR 各時間點病患數")
display(cdr_n_table)

In [ ]:
# 建立趨勢圖資料
""" 
同一病患在同一時間區間可能有多筆紀錄。為避免回診較密集者權重過高，先計算：

每位病患、每個時間區間的平均分數

再計算三組平均。"""

def create_plot_summary(df, outcome):
    # 每位病患在每個時間區間只貢獻一個平均值
    patient_period = (
        df.groupby(
            [
                "patient_id",
                "bio_category",
                "time_category_number"
            ],
            observed=True,
            as_index=False
        )
        .agg(
            patient_period_value=(outcome, "mean")
        )
    )

    summary = (
        patient_period
        .groupby(
            [
                "bio_category",
                "time_category_number"
            ],
            observed=True,
            as_index=False
        )
        .agg(
            mean=("patient_period_value", "mean"),
            sd=("patient_period_value", "std"),
            n=("patient_period_value", "count")
        )
    )

    summary["se"] = summary["sd"] / np.sqrt(summary["n"])
    summary["lower_95"] = summary["mean"] - 1.96 * summary["se"]
    summary["upper_95"] = summary["mean"] + 1.96 * summary["se"]

    summary["time_mid_years"] = (
        (summary["time_category_number"] - 0.5)
        * BIN_MONTHS / 12
    )

    return summary


plot_mmse = create_plot_summary(
    mmse_long,
    "mmse"
)

plot_casi = create_plot_summary(
    casi_long,
    "casi_total"
)

plot_cdr = create_plot_summary(
    mmse_long,
    "cdr_global"
)

display(plot_mmse.head())

In [ ]:
# 畫圖函數
def plot_longitudinal_trend(
    summary_df,
    outcome_label,
    title,
    filename,
    y_limits=None,
    annotate_n=True
):
    fig, ax = plt.subplots(figsize=(10, 6))

    for group in GROUP_ORDER:
        group_df = (
            summary_df[
                summary_df["bio_category"] == group
            ]
            .sort_values("time_mid_years")
        )

        if group_df.empty:
            continue

        ax.plot(
            group_df["time_mid_years"],
            group_df["mean"],
            marker="o",
            linewidth=2,
            label=group
        )

        ax.fill_between(
            group_df["time_mid_years"].to_numpy(dtype=float),
            group_df["lower_95"].to_numpy(dtype=float),
            group_df["upper_95"].to_numpy(dtype=float),
            alpha=0.15
        )

        if annotate_n:
            for _, row in group_df.iterrows():
                if pd.notna(row["mean"]):
                    ax.annotate(
                        f"n={int(row['n'])}",
                        (
                            row["time_mid_years"],
                            row["mean"]
                        ),
                        textcoords="offset points",
                        xytext=(0, 7),
                        ha="center",
                        fontsize=8
                    )

    ax.set_title(title, fontsize=14)
    ax.set_xlabel("距離各量表第一次測量之追蹤時間（年）")
    ax.set_ylabel(outcome_label)

    if y_limits is not None:
        ax.set_ylim(y_limits)

    ax.legend(title="Biological category")
    ax.grid(alpha=0.25)
    fig.tight_layout()

    output_path = FIGURE_DIR / filename
    fig.savefig(
        output_path,
        dpi=300,
        bbox_inches="tight"
    )

    plt.show()

    print("圖片已儲存：", output_path.resolve())

In [ ]:
# MMSE 縱向趨勢圖

plot_longitudinal_trend(
    summary_df=plot_mmse,
    outcome_label="MMSE 平均分數",
    title="AD、non-AD 與 CU 三組之 MMSE 縱向變化趨勢",
    filename="MMSE_longitudinal_trend.png",
    y_limits=(0, 30)
)



In [ ]:
# CASI 縱向趨勢圖

plot_longitudinal_trend(
    summary_df=plot_casi,
    outcome_label="CASI 平均分數",
    title="AD、non-AD 與 CU 三組之 CASI 縱向變化趨勢",
    filename="CASI_longitudinal_trend.png",
    y_limits=(0, 100)
)



In [ ]:
# CDR 縱向趨勢圖

plot_longitudinal_trend(
    summary_df=plot_cdr,
    outcome_label="CDR global 平均分數",
    title="AD、non-AD 與 CU 三組之 CDR 縱向變化趨勢",
    filename="CDR_longitudinal_trend.png",
    y_limits=(0, 3)
)

In [ ]:
# 製作追蹤結構表

def patient_followup_summary(df, prefix):
    patient_level = (
        df.groupby(
            ["patient_id", "bio_category"],
            observed=True
        )
        .agg(
            visit_n=("visit_date", "nunique"),
            followup_years=("time_since_baseline_years", "max")
        )
        .reset_index()
    )

    rows = []

    for variable, label in [
        ("visit_n", f"{prefix} 測量次數"),
        ("followup_years", f"{prefix} 追蹤時間（年）")
    ]:
        row = {
            "指標": label,
            "整體": format_median_iqr(
                patient_level[variable]
            )
        }

        for group in GROUP_ORDER:
            values = patient_level.loc[
                patient_level["bio_category"] == group,
                variable
            ]

            row[group] = format_median_iqr(values)

        row["p 值"] = format_pvalue(
            kruskal_pvalue(
                patient_level.rename(
                    columns={"bio_category": "bio_category"}
                ),
                variable
            )
        )

        rows.append(row)

    return rows


table_followup_rows = []

table_followup_rows.extend(
    patient_followup_summary(
        mmse_long,
        "MMSE/CDR"
    )
)

table_followup_rows.extend(
    patient_followup_summary(
        casi_long,
        "CASI"
    )
)

table_followup = pd.DataFrame(table_followup_rows)

display(table_followup)

In [ ]:
# 匯出所有表格與畫圖資料 
excel_output = OUTPUT_DIR / "AD_nonAD_CU_descriptive_statistics.xlsx"

with pd.ExcelWriter(
    excel_output,
    engine="openpyxl"
) as writer:

    group_count_table.to_excel(
        writer,
        sheet_name="group_counts",
        index=False
    )

    table1.to_excel(
        writer,
        sheet_name="Table_4_1_baseline",
        index=False
    )

    table_mmse.to_excel(
        writer,
        sheet_name="Table_4_2_MMSE",
        index=False
    )

    table_casi.to_excel(
        writer,
        sheet_name="Table_4_3_CASI",
        index=False
    )

    table_cdr.to_excel(
        writer,
        sheet_name="Table_4_4_CDR",
        index=False
    )

    table_cdr_sob.to_excel(
        writer,
        sheet_name="CDR_SOB_supplement",
        index=False
    )

    table_followup.to_excel(
        writer,
        sheet_name="Table_4_5_followup",
        index=False
    )

    mmse_n_table.to_excel(
        writer,
        sheet_name="MMSE_timepoint_N"
    )

    casi_n_table.to_excel(
        writer,
        sheet_name="CASI_timepoint_N"
    )

    cdr_n_table.to_excel(
        writer,
        sheet_name="CDR_timepoint_N"
    )

    plot_mmse.to_excel(
        writer,
        sheet_name="MMSE_plot_data",
        index=False
    )

    plot_casi.to_excel(
        writer,
        sheet_name="CASI_plot_data",
        index=False
    )

    plot_cdr.to_excel(
        writer,
        sheet_name="CDR_plot_data",
        index=False
    )

print("Excel 已輸出：", excel_output.resolve())

In [ ]:
# 輸出 stack data
# 後續 GEE 和 Mixed Effect Model 可以直接接這兩份資料

mmse_stack_output = OUTPUT_DIR / "MMSE_CDR_longitudinal_stack.csv"
casi_stack_output = OUTPUT_DIR / "CASI_longitudinal_stack.csv"

mmse_long.to_csv(
    mmse_stack_output,
    index=False,
    encoding="utf-8-sig"
)

casi_long.to_csv(
    casi_stack_output,
    index=False,
    encoding="utf-8-sig"
)

print("MMSE/CDR stack data：", mmse_stack_output.resolve())
print("CASI stack data：", casi_stack_output.resolve())

In [ ]:
# 最後品質檢查

print("========== 最終結果檢查 ==========")

print("\n1. 表 4.1 病患數")
display(
    baseline_df.groupby(
        "bio_category",
        observed=True
    )["patient_id"]
    .nunique()
    .to_frame("patient_n")
)

print("\n2. MMSE 可用觀測")
display(
    mmse_long.groupby(
        "bio_category",
        observed=True
    )["mmse"]
    .count()
    .to_frame("MMSE_available_n")
)

print("\n3. CASI 可用觀測")
display(
    casi_long.groupby(
        "bio_category",
        observed=True
    )["casi_total"]
    .count()
    .to_frame("CASI_available_n")
)

print("\n4. CDR 可用觀測")
display(
    mmse_long.groupby(
        "bio_category",
        observed=True
    )["cdr_global"]
    .count()
    .to_frame("CDR_available_n")
)

print("\n5. 輸出檔案")
for path in OUTPUT_DIR.rglob("*"):
    if path.is_file():
        print(path)